In [ ]:

!pip install gensim nltk scikit-learn tensorflow pandas numpy matplotlib seaborn


In [ ]:

import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

import gensim.downloader as api

nltk.download('stopwords')


In [ ]:

from google.colab import files
uploaded = files.upload()


In [ ]:

df = pd.read_csv('Book_review.csv')

print(df.head())
print(df.shape)


In [ ]:

df['summary'] = df['summary'].fillna('')
df['reviewText'] = df['reviewText'].fillna('')

df['text'] = df['summary'] + ' ' + df['reviewText']

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['text'] = df['text'].apply(clean_text)

print(df[['text']].head())


In [ ]:

def sentiment_label(rating):
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

df['sentiment'] = df['rating'].apply(sentiment_label)

print(df['sentiment'].value_counts())


In [ ]:

MAX_WORDS = 20000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(df['text'])

sequences = tokenizer.texts_to_sequences(df['text'])

X = pad_sequences(sequences, maxlen=MAX_LEN)

y = to_categorical(df['sentiment'], num_classes=3)

print(X.shape)
print(y.shape)


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)


In [ ]:

rnn_model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    SimpleRNN(72),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])

rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

rnn_model.summary()


In [ ]:

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)


In [ ]:

loss_rnn, acc_rnn = rnn_model.evaluate(X_test, y_test)

print('Simple RNN Accuracy:', acc_rnn)
print('Simple RNN Loss:', loss_rnn)


In [ ]:

bilstm_model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    Bidirectional(LSTM(80)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])

bilstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

bilstm_model.summary()


In [ ]:

history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)


In [ ]:

loss_bilstm, acc_bilstm = bilstm_model.evaluate(X_test, y_test)

print('BiLSTM Accuracy:', acc_bilstm)
print('BiLSTM Loss:', loss_bilstm)
